In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set
import re

@dataclass
class RaceMetadata:
    """Metadata for D&D races"""
    name: str
    size: str = ""
    speed: str = ""
    languages: List[str] = field(default_factory=list)
    ability_increases: Dict[str, int] = field(default_factory=dict)
    age_range: str = ""
    alignment_tendency: str = ""
    has_darkvision: bool = False
    darkvision_range: int = 0
    subrace: Optional[str] = None
    parent_race: Optional[str] = None

@dataclass
class RaceChunk:
    """Individual chunk of race information"""
    content: str
    chunk_type: str  # 'description', 'traits', 'subrace', 'names', 'culture'
    metadata: RaceMetadata
    tokens_estimate: int = 0
    tags: Set[str] = field(default_factory=set)

class RaceTextChunker:
    """Handles chunking of race text content"""

    def __init__(self):
        self.max_tokens = 400
        self.overlap_tokens = 50

    def estimate_tokens(self, text: str) -> int:
        """Rough token estimation (1 token ≈ 4 characters)"""
        return len(text) // 4

    def chunk_long_text(self, text: str, chunk_type: str = "description") -> List[str]:
        """Split long text into smaller chunks with overlap"""
        if self.estimate_tokens(text) <= self.max_tokens:
            return [text]

        chunks = []
        sentences = re.split(r'(?<=[.!?])\s+', text)

        current_chunk = ""
        for sentence in sentences:
            test_chunk = current_chunk + " " + sentence if current_chunk else sentence

            if self.estimate_tokens(test_chunk) > self.max_tokens and current_chunk:
                chunks.append(current_chunk.strip())
                # Add overlap from last few words
                words = current_chunk.split()
                overlap = " ".join(words[-10:]) if len(words) > 10 else current_chunk
                current_chunk = overlap + " " + sentence
            else:
                current_chunk = test_chunk

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks

print("Race data models ready!")

Race data models ready!


In [2]:
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import asdict

class DNDRaceParser:
    """Parser for D&D race information from manual text"""

    def __init__(self):
        self.race_patterns = {
            'ability_increase': r'Ability Score Increase\.\s*([^.]+)',
            'age': r'Age\.\s*([^.]+(?:\.|$))',
            'alignment': r'Alignment\.\s*([^.]+(?:\.|$))',
            'size': r'Size\.\s*([^.]+(?:\.|$))',
            'speed': r'Speed\.\s*([^.]+(?:\.|$))',
            'darkvision': r'Darkvision\.\s*([^.]+(?:\.|$))',
            'languages': r'Languages\.\s*([^.]+(?:\.|$))',
            'subrace': r'Subrace\.\s*([^.]+(?:\.|$))',
        }

        self.trait_patterns = [
            r'([A-Z][a-z\s]+)\.\s*([^.]+\.)',  # General trait pattern
            r'(Superior Darkvision|Sunlight Sensitivity|Drow Magic|Drow Weapon Training)\.\s*([^.]+\.)',
            r'(Keen Senses|Fey Ancestry|Trance)\.\s*([^.]+\.)',
            r'(Dwarven Resilience|Dwarven Combat Training|Tool Proficiency|Stonecunning)\.\s*([^.]+\.)',
        ]

    def extract_race_sections(self, text: str) -> List[Dict]:
        """Extract individual race sections from the manual text"""
        # Look for race headers (all caps race names)
        race_headers = re.findall(r'\n([A-Z]{3,})\n', text)

        # Split text by race headers
        races = []
        sections = re.split(r'\n[A-Z]{3,}\n', text)

        for i, header in enumerate(race_headers):
            if i + 1 < len(sections):
                race_data = {
                    'name': header.title(),
                    'content': sections[i + 1].strip()
                }
                races.append(race_data)

        return races

    def extract_subraces(self, race_content: str) -> List[Dict]:
        """Extract subrace information from race content"""
        subraces = []

        # Look for subrace headers (HILL DWARF, HIGH ELF, etc.)
        subrace_pattern = r'\n([A-Z\s]+(?:DWARF|ELF|HALFLING|GNOME))\n(.*?)(?=\n[A-Z\s]+(?:DWARF|ELF|HALFLING|GNOME)|\Z)'
        matches = re.findall(subrace_pattern, race_content, re.DOTALL)

        for match in matches:
            subrace_name = match[0].strip().title()
            subrace_content = match[1].strip()

            subraces.append({
                'name': subrace_name,
                'content': subrace_content
            })

        return subraces

    def parse_ability_increases(self, text: str) -> Dict[str, int]:
        """Parse ability score increases"""
        ability_increases = {}

        # Look for ability score patterns
        patterns = [
            r'Your (\w+) score increases by (\d+)',
            r'(\w+) score increases by (\d+)',
        ]

        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for ability, increase in matches:
                ability_increases[ability.lower()] = int(increase)

        return ability_increases

    def parse_darkvision(self, text: str) -> Tuple[bool, int]:
        """Parse darkvision information"""
        darkvision_match = re.search(r'darkvision.*?(\d+)\s*feet', text, re.IGNORECASE)
        if darkvision_match:
            return True, int(darkvision_match.group(1))
        return False, 0

    def extract_traits(self, text: str) -> List[Dict]:
        """Extract racial traits from text"""
        traits = []

        for pattern in self.trait_patterns:
            matches = re.findall(pattern, text, re.DOTALL)
            for trait_name, trait_desc in matches:
                traits.append({
                    'name': trait_name.strip(),
                    'description': trait_desc.strip()
                })

        return traits

    def parse_race_metadata(self, race_name: str, content: str) -> RaceMetadata:
        """Parse race content into metadata"""
        # Extract basic information
        ability_increases = self.parse_ability_increases(content)
        has_darkvision, darkvision_range = self.parse_darkvision(content)

        # Extract other attributes using patterns
        size = ""
        size_match = re.search(self.race_patterns['size'], content, re.IGNORECASE | re.DOTALL)
        if size_match:
            size = size_match.group(1).strip()

        speed = ""
        speed_match = re.search(self.race_patterns['speed'], content, re.IGNORECASE | re.DOTALL)
        if speed_match:
            speed = speed_match.group(1).strip()

        age_range = ""
        age_match = re.search(self.race_patterns['age'], content, re.IGNORECASE | re.DOTALL)
        if age_match:
            age_range = age_match.group(1).strip()

        alignment = ""
        alignment_match = re.search(self.race_patterns['alignment'], content, re.IGNORECASE | re.DOTALL)
        if alignment_match:
            alignment = alignment_match.group(1).strip()

        languages = []
        lang_match = re.search(self.race_patterns['languages'], content, re.IGNORECASE | re.DOTALL)
        if lang_match:
            lang_text = lang_match.group(1).strip()
            # Extract language names
            lang_patterns = [r'Common', r'Elvish', r'Dwarvish', r'Halfling', r'Giant', r'Gnomish']
            for pattern in lang_patterns:
                if re.search(pattern, lang_text, re.IGNORECASE):
                    languages.append(pattern)

        return RaceMetadata(
            name=race_name,
            size=size,
            speed=speed,
            languages=languages,
            ability_increases=ability_increases,
            age_range=age_range,
            alignment_tendency=alignment,
            has_darkvision=has_darkvision,
            darkvision_range=darkvision_range
        )

print("Race parser ready!")

Race parser ready!


In [3]:
from typing import List, Dict, Set
import re

class RaceChunkCreator:
    """Creates structured chunks from race information"""

    def __init__(self):
        self.chunker = RaceTextChunker()

    def create_race_chunks(self, race_name: str, race_content: str, metadata: RaceMetadata) -> List[RaceChunk]:
        """Create all chunks for a race"""
        chunks = []

        # 1. Main description chunk
        desc_chunks = self._create_description_chunks(race_name, race_content, metadata)
        chunks.extend(desc_chunks)

        # 2. Racial traits chunks
        traits_chunks = self._create_traits_chunks(race_name, race_content, metadata)
        chunks.extend(traits_chunks)

        # 3. Names and culture chunk
        names_chunk = self._create_names_chunk(race_name, race_content, metadata)
        if names_chunk:
            chunks.append(names_chunk)

        # 4. Mechanical summary chunk
        mechanical_chunk = self._create_mechanical_chunk(race_name, metadata)
        if mechanical_chunk:
            chunks.append(mechanical_chunk)

        return chunks

    def _create_description_chunks(self, race_name: str, content: str, metadata: RaceMetadata) -> List[RaceChunk]:
        """Create description chunks from race lore and background"""
        chunks = []

        # Extract descriptive paragraphs (before traits section)
        trait_start = re.search(r'(Ability Score Increase|Age\.|Size\.)', content, re.IGNORECASE)
        if trait_start:
            description_text = content[:trait_start.start()].strip()
        else:
            description_text = content[:500]  # Fallback to first 500 chars

        if not description_text:
            return chunks

        # Add race introduction
        intro_text = f"**{race_name}** - Core D&D Race\n\n{description_text}"

        # Split into manageable chunks if too long
        text_chunks = self.chunker.chunk_long_text(intro_text)

        for i, chunk_text in enumerate(text_chunks):
            tags = {f"race_{race_name.lower()}", "description", "lore"}

            chunks.append(RaceChunk(
                content=chunk_text,
                chunk_type="description",
                metadata=metadata,
                tokens_estimate=self.chunker.estimate_tokens(chunk_text),
                tags=tags
            ))

        return chunks

    def _create_traits_chunks(self, race_name: str, content: str, metadata: RaceMetadata) -> List[RaceChunk]:
        """Create chunks for racial traits"""
        chunks = []
        parser = DNDRaceParser()

        traits = parser.extract_traits(content)

        # Group related traits
        if traits:
            trait_text = f"**{race_name} Racial Traits:**\n\n"

            for trait in traits:
                trait_text += f"**{trait['name']}:** {trait['description']}\n\n"

            # Add mechanical summary
            trait_text += "**Core Mechanics:**\n"
            if metadata.ability_increases:
                increases = [f"{k.title()} +{v}" for k, v in metadata.ability_increases.items()]
                trait_text += f"- Ability Increases: {', '.join(increases)}\n"

            if metadata.has_darkvision:
                trait_text += f"- Darkvision: {metadata.darkvision_range} feet\n"

            if metadata.languages:
                trait_text += f"- Languages: {', '.join(metadata.languages)}\n"

            if metadata.speed:
                trait_text += f"- Speed: {metadata.speed}\n"

            if metadata.size:
                trait_text += f"- Size: {metadata.size}\n"

            tags = {f"race_{race_name.lower()}", "traits", "mechanics"}

            chunks.append(RaceChunk(
                content=trait_text,
                chunk_type="traits",
                metadata=metadata,
                tokens_estimate=self.chunker.estimate_tokens(trait_text),
                tags=tags
            ))

        return chunks

    def _create_names_chunk(self, race_name: str, content: str, metadata: RaceMetadata) -> Optional[RaceChunk]:
        """Create chunk for names and cultural information"""
        # Look for names section
        names_match = re.search(r'(NAMES?|NAMING)', content, re.IGNORECASE)
        if not names_match:
            return None

        # Extract names section
        names_start = names_match.start()
        names_text = content[names_start:names_start + 1000]  # Get reasonable chunk

        # Clean up and format
        names_content = f"**{race_name} Names and Culture:**\n\n{names_text.strip()}"

        tags = {f"race_{race_name.lower()}", "names", "culture", "character_creation"}

        return RaceChunk(
            content=names_content,
            chunk_type="names",
            metadata=metadata,
            tokens_estimate=self.chunker.estimate_tokens(names_content),
            tags=tags
        )

    def _create_mechanical_chunk(self, race_name: str, metadata: RaceMetadata) -> Optional[RaceChunk]:
        """Create a quick-reference mechanical chunk"""
        if not any([metadata.ability_increases, metadata.has_darkvision, metadata.languages]):
            return None

        content = f"**{race_name} - Quick Reference:**\n\n"

        # Ability increases
        if metadata.ability_increases:
            content += "**Ability Score Increases:**\n"
            for ability, increase in metadata.ability_increases.items():
                content += f"- {ability.title()}: +{increase}\n"
            content += "\n"

        # Key features
        content += "**Key Features:**\n"
        if metadata.has_darkvision:
            content += f"- Darkvision ({metadata.darkvision_range} ft)\n"

        if metadata.languages:
            content += f"- Languages: {', '.join(metadata.languages)}\n"

        if metadata.speed:
            content += f"- Base Speed: {metadata.speed}\n"

        if metadata.size:
            content += f"- Size Category: {metadata.size}\n"

        # Character creation guidance
        content += "\n**Character Creation Notes:**\n"
        content += f"- Age Range: {metadata.age_range}\n" if metadata.age_range else ""
        content += f"- Typical Alignment: {metadata.alignment_tendency}\n" if metadata.alignment_tendency else ""

        tags = {f"race_{race_name.lower()}", "quick_reference", "mechanics", "character_creation"}

        return RaceChunk(
            content=content,
            chunk_type="quick_reference",
            metadata=metadata,
            tokens_estimate=self.chunker.estimate_tokens(content),
            tags=tags
        )

    def create_subrace_chunks(self, parent_race: str, subrace_name: str, subrace_content: str) -> List[RaceChunk]:
        """Create chunks for subraces"""
        chunks = []

        # Create subrace metadata
        subrace_metadata = RaceMetadata(
            name=subrace_name,
            parent_race=parent_race,
            subrace=subrace_name
        )

        # Parse subrace-specific abilities
        parser = DNDRaceParser()
        ability_increases = parser.parse_ability_increases(subrace_content)
        subrace_metadata.ability_increases = ability_increases

        # Create subrace description chunk
        content = f"**{subrace_name}** (Subrace of {parent_race})\n\n{subrace_content}"

        tags = {f"race_{parent_race.lower()}", f"subrace_{subrace_name.lower()}", "subrace", "variant"}

        chunk = RaceChunk(
            content=content,
            chunk_type="subrace",
            metadata=subrace_metadata,
            tokens_estimate=self.chunker.estimate_tokens(content),
            tags=tags
        )

        chunks.append(chunk)
        return chunks

print("Race chunk creator ready!")

Race chunk creator ready!


In [4]:
import chromadb
from chromadb.config import Settings
import uuid
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional

class RaceChromaDBManager:
    """Manages D&D race data in ChromaDB"""

    def __init__(self, db_path: str = "./chromadb", model_name: str = "all-MiniLM-L6-v2"):
        self.db_path = db_path
        self.model_name = model_name
        self.client = chromadb.PersistentClient(path=db_path)

        print(f"🤖 Loading embedding model: {model_name}")
        self.embedding_model = SentenceTransformer(model_name)
        print("✅ Embedding model loaded!")

        self.collection = self.client.get_or_create_collection(
            name="dnd_races",
            metadata={"description": "D&D 5e race chunks for RAG system"}
        )

        print(f"📚 ChromaDB initialized at: {db_path}")
        print(f"   Collection: {self.collection.name}")
        print(f"   Current documents: {self.collection.count()}")

    def add_race_chunks(self, chunks: List[RaceChunk], batch_size: int = 50):
        """Add race chunks to ChromaDB"""
        if not chunks:
            print("❌ No chunks to add!")
            return

        print(f"📦 Adding {len(chunks)} chunks to ChromaDB...")

        # Prepare data for ChromaDB
        documents = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            # Create document text
            doc_text = chunk.content

            # Create metadata
            metadata = {
                "race_name": chunk.metadata.name,
                "chunk_type": chunk.chunk_type,
                "size": chunk.metadata.size,
                "speed": chunk.metadata.speed,
                "age_range": chunk.metadata.age_range,
                "alignment": chunk.metadata.alignment_tendency,
                "has_darkvision": chunk.metadata.has_darkvision,
                "darkvision_range": chunk.metadata.darkvision_range,
                "languages": ",".join(chunk.metadata.languages) if chunk.metadata.languages else "",
                "tags": ",".join(chunk.tags),
                "token_estimate": chunk.tokens_estimate,
                "parent_race": chunk.metadata.parent_race or "",
                "subrace": chunk.metadata.subrace or ""
            }

            # Add ability increases as separate fields
            for ability, increase in chunk.metadata.ability_increases.items():
                metadata[f"{ability}_increase"] = increase

            documents.append(doc_text)
            metadatas.append(metadata)
            ids.append(f"race_chunk_{i}_{uuid.uuid4().hex[:8]}")

        # Add to ChromaDB in batches
        for i in range(0, len(documents), batch_size):
            batch_end = min(i + batch_size, len(documents))

            print(f"   📤 Adding batch {i//batch_size + 1}/{(len(documents)-1)//batch_size + 1}")

            self.collection.add(
                documents=documents[i:batch_end],
                metadatas=metadatas[i:batch_end],
                ids=ids[i:batch_end]
            )

        print(f"✅ Successfully added {len(chunks)} race chunks to ChromaDB!")
        print(f"   Total documents in collection: {self.collection.count()}")

    def search_races(self, query: str, n_results: int = 5, **filter_kwargs) -> dict:
        """Search race information"""
        print(f"🔍 Searching races for: '{query}'")

        # Build where clause
        where_clause = {}

        if "race_name" in filter_kwargs:
            where_clause["race_name"] = {"$eq": filter_kwargs["race_name"]}
        if "chunk_type" in filter_kwargs:
            where_clause["chunk_type"] = {"$eq": filter_kwargs["chunk_type"]}
        if "has_darkvision" in filter_kwargs:
            where_clause["has_darkvision"] = {"$eq": filter_kwargs["has_darkvision"]}
        if "size" in filter_kwargs:
            where_clause["size"] = {"$eq": filter_kwargs["size"]}

        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where_clause if where_clause else None
        )

        print(f"   📋 Found {len(results['documents'][0])} results")
        return results

    def get_race_by_name(self, race_name: str) -> dict:
        """Get all chunks for a specific race"""
        results = self.collection.query(
            query_texts=[f"{race_name} race"],
            n_results=20
        )

        # Filter for exact race name matches
        filtered = {"documents": [[]], "metadatas": [[]], "distances": [[]], "ids": [[]]}

        if results["documents"][0]:
            for i, meta in enumerate(results["metadatas"][0]):
                if race_name.lower() in meta.get("race_name", "").lower():
                    filtered["documents"][0].append(results["documents"][0][i])
                    filtered["metadatas"][0].append(meta)
                    filtered["distances"][0].append(results["distances"][0][i])
                    filtered["ids"][0].append(results["ids"][0][i])

        return filtered

    def get_races_with_ability(self, ability: str, min_increase: int = 1) -> dict:
        """Find races that increase a specific ability"""
        field_name = f"{ability.lower()}_increase"

        # Since ChromaDB doesn't support numeric comparisons well, we'll search broadly
        results = self.collection.query(
            query_texts=[f"{ability} increase racial bonus"],
            n_results=50
        )

        # Manual filtering for ability increases
        filtered = {"documents": [[]], "metadatas": [[]], "distances": [[]], "ids": [[]]}

        if results["documents"][0]:
            for i, meta in enumerate(results["metadatas"][0]):
                if field_name in meta and meta[field_name] >= min_increase:
                    filtered["documents"][0].append(results["documents"][0][i])
                    filtered["metadatas"][0].append(meta)
                    filtered["distances"][0].append(results["distances"][0][i])
                    filtered["ids"][0].append(results["ids"][0][i])

        return filtered

    def get_collection_stats(self) -> dict:
        """Get statistics about the race collection"""
        total_docs = self.collection.count()

        if total_docs == 0:
            return {"total_documents": 0}

        # Sample some documents to analyze
        sample = self.collection.get(limit=min(100, total_docs))

        stats = {
            "total_documents": total_docs,
            "chunk_types": {},
            "races": set(),
            "subraces": set(),
            "abilities_increased": {}
        }

        if sample["metadatas"]:
            for metadata in sample["metadatas"]:
                # Count chunk types
                chunk_type = metadata.get("chunk_type", "unknown")
                stats["chunk_types"][chunk_type] = stats["chunk_types"].get(chunk_type, 0) + 1

                # Collect races
                race_name = metadata.get("race_name", "")
                if race_name:
                    stats["races"].add(race_name)

                # Collect subraces
                subrace = metadata.get("subrace", "")
                if subrace:
                    stats["subraces"].add(subrace)

                # Count ability increases
                for key, value in metadata.items():
                    if key.endswith("_increase") and value > 0:
                        ability = key.replace("_increase", "").title()
                        stats["abilities_increased"][ability] = stats["abilities_increased"].get(ability, 0) + 1

        stats["races"] = list(stats["races"])
        stats["subraces"] = list(stats["subraces"])
        return stats

    def preview_chunks(self, race_name: Optional[str] = None, limit: int = 3):
        """Preview chunks in the collection"""
        if race_name:
            results = self.get_race_by_name(race_name)
        else:
            results = self.collection.get(limit=limit)

        if not results["documents"] or not results["documents"][0]:
            print("No documents found")
            return

        docs = results["documents"][0] if isinstance(results["documents"][0], list) else results["documents"]
        metas = results["metadatas"][0] if isinstance(results["metadatas"][0], list) else results["metadatas"]

        for i, (doc, meta) in enumerate(zip(docs[:limit], metas[:limit])):
            print(f"\n{'='*60}")
            print(f"Chunk {i+1}: {meta.get('race_name', 'Unknown')} - {meta.get('chunk_type', 'Unknown')}")
            print(f"Tags: {meta.get('tags', '')}")
            print(f"Tokens: {meta.get('token_estimate', 0)}")
            print("-" * 40)
            print(doc[:300] + "..." if len(doc) > 300 else doc)

print("Race ChromaDB manager ready!")

/Users/alexchilton/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Race ChromaDB manager ready!


In [5]:
# Complete Race Processing Pipeline
# This combines all the components to process pages 18-46 of the Player's Handbook

class RaceProcessingPipeline:
    """Complete pipeline for processing D&D race information"""

    def __init__(self):
        self.parser = DNDRaceParser()
        self.chunk_creator = RaceChunkCreator()
        self.stats = {}

    def process_races(self, race_text: str) -> List[RaceChunk]:
        """Process race text and create chunks"""
        print("🎯 Starting race processing pipeline...")

        # Reset stats
        self.stats = {
            "races_processed": 0,
            "subraces_processed": 0,
            "total_chunks": 0,
            "chunk_types": {},
            "average_tokens": 0
        }

        all_chunks = []

        # Extract race sections
        print("📖 Extracting race sections...")
        race_sections = self.parser.extract_race_sections(race_text)

        print(f"Found {len(race_sections)} races to process")

        for race_data in race_sections:
            race_name = race_data['name']
            race_content = race_data['content']

            print(f"   Processing: {race_name}")

            # Parse race metadata
            metadata = self.parser.parse_race_metadata(race_name, race_content)

            # Create main race chunks
            race_chunks = self.chunk_creator.create_race_chunks(race_name, race_content, metadata)
            all_chunks.extend(race_chunks)

            # Process subraces
            subraces = self.parser.extract_subraces(race_content)
            for subrace_data in subraces:
                subrace_name = subrace_data['name']
                subrace_content = subrace_data['content']

                print(f"     Subrace: {subrace_name}")
                subrace_chunks = self.chunk_creator.create_subrace_chunks(
                    race_name, subrace_name, subrace_content
                )
                all_chunks.extend(subrace_chunks)
                self.stats["subraces_processed"] += 1

            self.stats["races_processed"] += 1

        # Update stats
        self.stats["total_chunks"] = len(all_chunks)

        # Count chunk types
        for chunk in all_chunks:
            chunk_type = chunk.chunk_type
            self.stats["chunk_types"][chunk_type] = self.stats["chunk_types"].get(chunk_type, 0) + 1

        # Calculate average tokens
        if all_chunks:
            total_tokens = sum(chunk.tokens_estimate for chunk in all_chunks)
            self.stats["average_tokens"] = total_tokens // len(all_chunks)

        print(f"✅ Processed {self.stats['races_processed']} races and {self.stats['subraces_processed']} subraces")
        print(f"📦 Created {self.stats['total_chunks']} total chunks")

        return all_chunks

    def get_chunk_statistics(self) -> dict:
        """Get processing statistics"""
        return self.stats.copy()

    def preview_chunks(self, chunks: List[RaceChunk], limit: int = 3):
        """Preview created chunks"""
        for i, chunk in enumerate(chunks[:limit]):
            print(f"\n{'='*60}")
            print(f"Chunk {i+1}: {chunk.metadata.name} - {chunk.chunk_type}")
            print(f"Tags: {', '.join(chunk.tags)}")
            print(f"Tokens: {chunk.tokens_estimate}")
            print("-" * 40)
            content_preview = chunk.content[:300] + "..." if len(chunk.content) > 300 else chunk.content
            print(content_preview)

# Utility function to load race text from file
def load_race_text(file_path: str) -> str:
    """Load race text from file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return ""

print("Race processing pipeline ready!")

Race processing pipeline ready!


In [7]:
import pdfplumber
import re
from typing import Optional, List, Dict

class RacePDFExtractor:
    """Extract race information from PDF files"""

    def __init__(self):
        self.page_range = (18, 46)  # Pages 18-46 for races

    def extract_pdf_text(self, pdf_path: str, start_page: int = 18, end_page: int = 46) -> str:
        """Extract text from specific pages of PDF"""
        try:
            print(f"📖 Extracting text from PDF: {pdf_path}")
            print(f"   Pages: {start_page}-{end_page}")

            extracted_text = ""

            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                print(f"   Total pages in PDF: {total_pages}")

                # Adjust page numbers (PDF is 0-indexed, but we use 1-indexed)
                start_idx = start_page - 1
                end_idx = min(end_page, total_pages)

                for page_num in range(start_idx, end_idx):
                    if page_num < len(pdf.pages):
                        page = pdf.pages[page_num]
                        page_text = page.extract_text()

                        if page_text:
                            # Clean up the text
                            cleaned_text = self.clean_extracted_text(page_text)
                            extracted_text += f"\n--- PAGE {page_num + 1} ---\n{cleaned_text}\n"
                            print(f"   ✅ Extracted page {page_num + 1}")
                        else:
                            print(f"   ❌ No text found on page {page_num + 1}")
                    else:
                        print(f"   ❌ Page {page_num + 1} not found")

            print(f"✅ Successfully extracted {len(extracted_text)} characters")
            return extracted_text

        except Exception as e:
            print(f"❌ Error extracting PDF: {str(e)}")
            return ""

    def clean_extracted_text(self, text: str) -> str:
        """Clean and normalize extracted PDF text"""
        if not text:
            return ""

        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)

        # Fix common PDF extraction issues
        text = text.replace('\n', ' ')
        text = text.replace('\r', ' ')

        # Remove page numbers and headers/footers (common patterns)
        text = re.sub(r'\b\d+\s*$', '', text)  # Page numbers at end
        text = re.sub(r'^PART\s+I+\s*\|\s*RACES\s*', '', text, flags=re.IGNORECASE)

        # Fix common character encoding issues


        # Normalize spaces
        text = ' '.join(text.split())

        return text.strip()

    def extract_pages_separately(self, pdf_path: str, start_page: int = 18, end_page: int = 46) -> Dict[int, str]:
        """Extract text from each page separately for better processing"""
        try:
            pages_text = {}

            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                start_idx = start_page - 1
                end_idx = min(end_page, total_pages)

                for page_num in range(start_idx, end_idx):
                    if page_num < len(pdf.pages):
                        page = pdf.pages[page_num]
                        page_text = page.extract_text()

                        if page_text:
                            cleaned_text = self.clean_extracted_text(page_text)
                            pages_text[page_num + 1] = cleaned_text

            return pages_text

        except Exception as e:
            print(f"❌ Error extracting PDF pages: {str(e)}")
            return {}

    def save_extracted_text(self, text: str, filename: str = "extracted_races.txt"):
        """Save extracted text to file for inspection"""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(text)
            print(f"💾 Extracted text saved to {filename}")
        except Exception as e:
            print(f"❌ Error saving text: {str(e)}")

    def validate_race_content(self, text: str) -> bool:
        """Validate that the extracted text contains race information"""
        race_indicators = [
            r'\belf\b', r'\bdwarf\b', r'\bhalfling\b', r'\bdragonborn\b',
            r'\bgnome\b', r'\bhuman\b', r'\btiefl?ing\b', r'\bhalf-?elf\b',
            r'ability score increase', r'darkvision', r'racial trait'
        ]

        matches = 0
        for pattern in race_indicators:
            if re.search(pattern, text, re.IGNORECASE):
                matches += 1

        is_valid = matches >= 3
        print(f"📋 Content validation: {matches}/10 race indicators found")
        print(f"   Valid race content: {'✅ Yes' if is_valid else '❌ No'}")

        return is_valid

    def find_race_sections(self, text: str) -> List[Dict]:
        """Find race section boundaries in the text"""
        # Look for race names that are typically in all caps
        race_patterns = [
            r'\b(DRAGONBORN)\b',
            r'\b(DWARF|DWARVES)\b',
            r'\b(ELF|ELVES)\b',
            r'\b(GNOME)\b',
            r'\b(HALF-ELF)\b',
            r'\b(HALFLING)\b',
            r'\b(HALF-ORC)\b',
            r'\b(HUMAN)\b',
            r'\b(TIEFLING)\b'
        ]

        found_races = []

        for pattern in race_patterns:
            matches = list(re.finditer(pattern, text, re.IGNORECASE))
            for match in matches:
                race_name = match.group(1).title()
                start_pos = match.start()

                found_races.append({
                    'name': race_name,
                    'start_position': start_pos,
                    'pattern_match': match.group(0)
                })

        # Sort by position in text
        found_races.sort(key=lambda x: x['start_position'])

        print(f"🔍 Found {len(found_races)} potential race sections:")
        for race in found_races:
            print(f"   - {race['name']} at position {race['start_position']}")

        return found_races

def process_race_pdf(pdf_path: str) -> str:
    """Complete pipeline: extract PDF -> clean -> validate -> return text"""

    print("🚀 Starting Race PDF Processing Pipeline")
    print("=" * 50)

    extractor = RacePDFExtractor()

    # Extract text from PDF
    raw_text = extractor.extract_pdf_text(pdf_path)

    if not raw_text:
        print("❌ Failed to extract text from PDF")
        return ""

    # Save raw text as backup
    extractor.save_extracted_text(raw_text, "extracted_races_raw.txt")

    # Validate content
    if not extractor.validate_race_content(raw_text):
        print("⚠️  Warning: Extracted content may not contain complete race information")

    # Find race sections
    race_sections = extractor.find_race_sections(raw_text)

    if race_sections:
        print(f"✅ Successfully identified {len(race_sections)} races in the text")
    else:
        print("⚠️  No clear race sections found - you may need to adjust the extraction")

    return raw_text

print("Race PDF extractor ready!")
print("Usage: race_text = process_race_pdf('path/to/players_handbook.pdf')")

Race PDF extractor ready!
Usage: race_text = process_race_pdf('path/to/players_handbook.pdf')


In [8]:
# D&D Race Processing Pipeline - Main Execution
# Run this cell to process races and add them to ChromaDB

# Configuration
RACE_PDF_PATH = "Dungeons  Dragons 5e Players Handbook (Wizards RPG Team Wyatt James, Schwalb Robert J etc.) (Z-Library).pdf"  # Update this to your PDF file path
RACE_TEXT_PATH = None  # Set this if you have a text file instead
DB_PATH = "./chromadb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print("🚀 Starting D&D Race Processing Pipeline")
print("=" * 50)

# Step 1: Load race text (from PDF or text file)
race_text = None

if RACE_PDF_PATH:
    print("📖 Extracting race text from PDF...")
    race_text = process_race_pdf(RACE_PDF_PATH)
    if race_text:
        print(f"✅ Extracted {len(race_text)} characters from PDF")
        print(f"   Preview: {race_text[:200]}...")
    else:
        print("❌ Failed to extract text from PDF. Please check the file path.")

elif RACE_TEXT_PATH:
    print("📖 Loading race text from text file...")
    race_text = load_race_text(RACE_TEXT_PATH)
    if race_text:
        print(f"✅ Loaded {len(race_text)} characters of race text")
        print(f"   Preview: {race_text[:100]}...")
    else:
        print("❌ Failed to load race text. Please check the file path.")

else:
    print("❌ No race file specified. Please set RACE_PDF_PATH or RACE_TEXT_PATH.")

# Step 2: Process races if text was loaded
if race_text:
    print("\n🎯 Processing races from text...")
    pipeline = RaceProcessingPipeline()
    chunks = pipeline.process_races(race_text)

    # Show statistics
    print("\n📊 Processing Statistics:")
    stats = pipeline.get_chunk_statistics()
    for key, value in stats.items():
        print(f"   {key}: {value}")

    # Preview some chunks
    print("\n👀 Preview of created chunks:")
    pipeline.preview_chunks(chunks, limit=2)

    print(f"\n✅ Successfully processed {stats.get('races_processed', 0)} races!")
    print(f"   🏃 Subraces: {stats.get('subraces_processed', 0)}")
    print(f"   📦 Total chunks: {stats.get('total_chunks', 0)}")
    print(f"   🎯 Average chunk size: {stats.get('average_tokens', 0)} tokens")

else:
    print("\n❌ Skipping processing due to missing race text")
    chunks = []

🚀 Starting D&D Race Processing Pipeline
📖 Extracting race text from PDF...
🚀 Starting Race PDF Processing Pipeline
📖 Extracting text from PDF: Dungeons  Dragons 5e Players Handbook (Wizards RPG Team Wyatt James, Schwalb Robert J etc.) (Z-Library).pdf
   Pages: 18-46
   Total pages in PDF: 319
   ✅ Extracted page 18
   ✅ Extracted page 19
   ✅ Extracted page 20
   ✅ Extracted page 21
   ✅ Extracted page 22
   ✅ Extracted page 23
   ✅ Extracted page 24
   ✅ Extracted page 25
   ❌ No text found on page 26
   ✅ Extracted page 27
   ✅ Extracted page 28
   ✅ Extracted page 29
   ✅ Extracted page 30
   ✅ Extracted page 31
   ✅ Extracted page 32
   ✅ Extracted page 33
   ✅ Extracted page 34
   ✅ Extracted page 35
   ✅ Extracted page 36
   ✅ Extracted page 37
   ✅ Extracted page 38
   ✅ Extracted page 39
   ✅ Extracted page 40
   ✅ Extracted page 41
   ✅ Extracted page 42
   ✅ Extracted page 43
   ✅ Extracted page 44
   ❌ No text found on page 45
   ✅ Extracted page 46
✅ Successfully extracted 

In [9]:
# Improved Race Parser specifically for D&D PDF text format

import re
from typing import List, Dict, Optional, Tuple

class ImprovedDNDRaceParser:
    """Enhanced parser that handles actual D&D PDF text format"""

    def __init__(self):
        # Race names as they appear in the PDF headers
        self.race_names = [
            'DRAGONBORN', 'DWARF', 'ELF', 'GNOME',
            'HALF-ELF', 'HALFLING', 'HALF-ORC', 'HUMAN', 'TIEFLING'
        ]

        self.subrace_patterns = [
            r'HILL DWARF', r'MOUNTAIN DWARF', r'DUERGAR',
            r'HIGH ELF', r'WOOD ELF', r'DARK ELF \(DROW\)', r'DROW',
            r'FOREST GNOME', r'ROCK GNOME', r'DEEP GNOME',
            r'LIGHTFOOT', r'STOUT', r'GHOSTWISE'
        ]

    def extract_race_sections_improved(self, text: str) -> List[Dict]:
        """Extract race sections using actual PDF text patterns"""
        race_sections = []

        # Clean the text first
        text = self._clean_pdf_text(text)

        # Look for race headers more specifically
        for race_name in self.race_names:
            # Find race section start
            race_pattern = rf'\b{race_name}\b(?!\s+(?:at position|TRAITS))'
            matches = list(re.finditer(race_pattern, text, re.IGNORECASE))

            for match in matches:
                start_pos = match.start()

                # Look for content that follows this pattern
                # Check if this looks like a race header (not just a mention)
                context_before = text[max(0, start_pos-100):start_pos]
                context_after = text[start_pos:start_pos+500]

                # Skip if this appears to be just a mention in narrative text
                if self._is_race_header(race_name, context_before, context_after):
                    # Find the end of this race section
                    end_pos = self._find_race_section_end(text, start_pos, race_name)

                    race_content = text[start_pos:end_pos].strip()

                    if len(race_content) > 200:  # Ensure it's substantial content
                        race_sections.append({
                            'name': race_name.title(),
                            'content': race_content,
                            'start_pos': start_pos,
                            'end_pos': end_pos
                        })
                        print(f"   Found {race_name.title()} section: {len(race_content)} characters")
                        break  # Take first good match for this race

        return race_sections

    def _clean_pdf_text(self, text: str) -> str:
        """Clean PDF text for better parsing"""
        # Remove page markers
        text = re.sub(r'--- PAGE \d+ ---', '', text)

        # Fix common PDF extraction issues
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)  # Add spaces between words
        text = re.sub(r'(\d+:)', r'\n\1', text)  # New lines after chapter numbers

        # Clean up excessive whitespace
        text = re.sub(r'\s+', ' ', text)

        return text

    def _is_race_header(self, race_name: str, context_before: str, context_after: str) -> bool:
        """Determine if this is actually a race section header"""

        # Look for indicators this is a section header
        header_indicators = [
            'CHAPTER', 'RACES', 'RACIAL TRAITS',
            'Ability Score Increase', 'Age.', 'Size.', 'Speed.'
        ]

        # Check context after the race name
        for indicator in header_indicators:
            if indicator in context_after[:300]:
                return True

        # Check if it's surrounded by other formatting that suggests a header
        if (len(context_before) < 50 or  # Near start of section
            'CHAPTER' in context_before[-50:] or  # After chapter marker
            race_name.upper() == context_after.split()[0]):  # First word
            return True

        return False

    def _find_race_section_end(self, text: str, start_pos: int, current_race: str) -> int:
        """Find where the current race section ends"""

        # Look for the next race header
        next_race_pos = len(text)

        for race_name in self.race_names:
            if race_name != current_race:
                next_match = re.search(rf'\b{race_name}\b', text[start_pos + 100:])
                if next_match:
                    candidate_pos = start_pos + 100 + next_match.start()

                    # Verify this is actually a new race header
                    context = text[candidate_pos:candidate_pos + 200]
                    if self._is_race_header(race_name, "", context):
                        next_race_pos = min(next_race_pos, candidate_pos)

        return next_race_pos

    def extract_race_metadata_improved(self, race_name: str, content: str) -> RaceMetadata:
        """Extract metadata from race content with better patterns"""

        # Initialize metadata
        metadata = RaceMetadata(name=race_name)

        # Extract ability score increases
        ability_pattern = r'Your (\w+) score increases by (\d+)'
        ability_matches = re.findall(ability_pattern, content, re.IGNORECASE)
        for ability, increase in ability_matches:
            metadata.ability_increases[ability.lower()] = int(increase)

        # Extract size
        size_match = re.search(r'Size\.\s*([^.]*?)\.', content, re.IGNORECASE | re.DOTALL)
        if size_match:
            metadata.size = size_match.group(1).strip()

        # Extract speed
        speed_match = re.search(r'Speed\.\s*([^.]*?)\.', content, re.IGNORECASE | re.DOTALL)
        if speed_match:
            metadata.speed = speed_match.group(1).strip()

        # Extract darkvision
        darkvision_match = re.search(r'darkvision.*?(\d+)\s*feet', content, re.IGNORECASE)
        if darkvision_match:
            metadata.has_darkvision = True
            metadata.darkvision_range = int(darkvision_match.group(1))

        # Extract age information
        age_match = re.search(r'Age\.\s*([^.]*?)\.', content, re.IGNORECASE | re.DOTALL)
        if age_match:
            metadata.age_range = age_match.group(1).strip()

        # Extract alignment
        alignment_match = re.search(r'Alignment\.\s*([^.]*?)\.', content, re.IGNORECASE | re.DOTALL)
        if alignment_match:
            metadata.alignment_tendency = alignment_match.group(1).strip()

        # Extract languages
        lang_match = re.search(r'Languages\.\s*([^.]*?)\.', content, re.IGNORECASE | re.DOTALL)
        if lang_match:
            lang_text = lang_match.group(1)
            # Common D&D languages
            languages = []
            for lang in ['Common', 'Elvish', 'Dwarvish', 'Halfling', 'Draconic', 'Giant', 'Gnomish', 'Goblin', 'Orc']:
                if lang in lang_text:
                    languages.append(lang)
            metadata.languages = languages

        return metadata

    def extract_subraces_improved(self, race_content: str) -> List[Dict]:
        """Extract subraces with better pattern matching"""
        subraces = []

        for pattern in self.subrace_patterns:
            matches = list(re.finditer(pattern, race_content, re.IGNORECASE))

            for match in matches:
                subrace_name = match.group(0)
                start_pos = match.start()

                # Find content for this subrace
                # Look for the next subrace or end of section
                end_pos = len(race_content)
                for other_pattern in self.subrace_patterns:
                    if other_pattern != pattern:
                        next_match = re.search(other_pattern, race_content[start_pos + 10:])
                        if next_match:
                            end_pos = min(end_pos, start_pos + 10 + next_match.start())

                subrace_content = race_content[start_pos:end_pos].strip()

                if len(subrace_content) > 50:  # Ensure substantial content
                    subraces.append({
                        'name': subrace_name.title(),
                        'content': subrace_content
                    })

        return subraces

# Test function to debug the parsing
def debug_race_parsing(text: str, limit: int = 3):
    """Debug function to see what's happening in race parsing"""

    print("🔍 Debug: Race Parsing Analysis")
    print("=" * 50)

    parser = ImprovedDNDRaceParser()

    # Show first few race mentions
    print(f"📖 First {limit} race mentions in text:")
    for race in parser.race_names[:limit]:
        matches = list(re.finditer(rf'\b{race}\b', text, re.IGNORECASE))
        print(f"\n{race}:")
        for i, match in enumerate(matches[:3]):  # Show first 3 matches
            start = max(0, match.start() - 50)
            end = min(len(text), match.end() + 100)
            context = text[start:end].replace('\n', ' ')
            print(f"  Match {i+1}: ...{context}...")

    # Test the improved parser
    print(f"\n🎯 Testing improved race section extraction:")
    race_sections = parser.extract_race_sections_improved(text)

    print(f"Found {len(race_sections)} race sections:")
    for section in race_sections:
        print(f"- {section['name']}: {len(section['content'])} chars")

print("Improved race parser ready!")
print("Use: debug_race_parsing(race_text) to test the parser")

Improved race parser ready!
Use: debug_race_parsing(race_text) to test the parser


In [10]:
# Complete Race Processing Pipeline
# This combines all the components to process pages 18-46 of the Player's Handbook

class RaceProcessingPipeline:
    """Complete pipeline for processing D&D race information"""

    def __init__(self):
        self.parser = ImprovedDNDRaceParser()
        self.chunk_creator = RaceChunkCreator()
        self.stats = {}

    def process_races(self, race_text: str) -> List[RaceChunk]:
        """Process race text and create chunks"""
        print("🎯 Starting race processing pipeline...")

        # Reset stats
        self.stats = {
            "races_processed": 0,
            "subraces_processed": 0,
            "total_chunks": 0,
            "chunk_types": {},
            "average_tokens": 0
        }

        all_chunks = []

        # Extract race sections
        print("📖 Extracting race sections...")
        race_sections = self.parser.extract_race_sections_improved(race_text)

        print(f"Found {len(race_sections)} races to process")

        for race_data in race_sections:
            race_name = race_data['name']
            race_content = race_data['content']

            print(f"   Processing: {race_name}")

            # Parse race metadata
            metadata = self.parser.extract_race_metadata_improved(race_name, race_content)

            # Create main race chunks
            race_chunks = self.chunk_creator.create_race_chunks(race_name, race_content, metadata)
            all_chunks.extend(race_chunks)

            # Process subraces
            subraces = self.parser.extract_subraces_improved(race_content)
            for subrace_data in subraces:
                subrace_name = subrace_data['name']
                subrace_content = subrace_data['content']

                print(f"     Subrace: {subrace_name}")
                subrace_chunks = self.chunk_creator.create_subrace_chunks(
                    race_name, subrace_name, subrace_content
                )
                all_chunks.extend(subrace_chunks)
                self.stats["subraces_processed"] += 1

            self.stats["races_processed"] += 1

        # Update stats
        self.stats["total_chunks"] = len(all_chunks)

        # Count chunk types
        for chunk in all_chunks:
            chunk_type = chunk.chunk_type
            self.stats["chunk_types"][chunk_type] = self.stats["chunk_types"].get(chunk_type, 0) + 1

        # Calculate average tokens
        if all_chunks:
            total_tokens = sum(chunk.tokens_estimate for chunk in all_chunks)
            self.stats["average_tokens"] = total_tokens // len(all_chunks)

        print(f"✅ Processed {self.stats['races_processed']} races and {self.stats['subraces_processed']} subraces")
        print(f"📦 Created {self.stats['total_chunks']} total chunks")

        return all_chunks

    def get_chunk_statistics(self) -> dict:
        """Get processing statistics"""
        return self.stats.copy()

    def preview_chunks(self, chunks: List[RaceChunk], limit: int = 3):
        """Preview created chunks"""
        for i, chunk in enumerate(chunks[:limit]):
            print(f"\n{'='*60}")
            print(f"Chunk {i+1}: {chunk.metadata.name} - {chunk.chunk_type}")
            print(f"Tags: {', '.join(chunk.tags)}")
            print(f"Tokens: {chunk.tokens_estimate}")
            print("-" * 40)
            content_preview = chunk.content[:300] + "..." if len(chunk.content) > 300 else chunk.content
            print(content_preview)

# Utility function to load race text from file
def load_race_text(file_path: str) -> str:
    """Load race text from file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return ""

print("Race processing pipeline ready!")

Race processing pipeline ready!


In [11]:
# Debug and Test the Race Parsing
# Run this to see what's happening with the race extraction

# Test the debug function on your extracted text
if 'race_text' in locals() and race_text:
    print("🔍 Debugging race parsing with your PDF text...")
    debug_race_parsing(race_text, limit=5)

    print("\n" + "="*60)
    print("🎯 Testing improved processing pipeline...")

    # Create new pipeline with improved parser
    improved_pipeline = RaceProcessingPipeline()

    # Process races with improved parser
    improved_chunks = improved_pipeline.process_races(race_text)

    # Show results
    print("\n📊 Improved Processing Results:")
    improved_stats = improved_pipeline.get_chunk_statistics()
    for key, value in improved_stats.items():
        print(f"   {key}: {value}")

    if improved_chunks:
        print("\n👀 Preview of improved chunks:")
        improved_pipeline.preview_chunks(improved_chunks, limit=3)

        # Update the global chunks variable
        chunks = improved_chunks
        print(f"\n✅ Updated chunks variable with {len(chunks)} race chunks!")
    else:
        print("\n❌ Still no chunks created. Need to investigate further.")

        # Let's look at the raw text structure
        print("\n🔍 Raw text analysis:")
        print(f"Total text length: {len(race_text)}")
        print(f"First 500 chars: {race_text[:500]}")

        # Look for any race names in the text
        race_mentions = {}
        for race in ['DRAGONBORN', 'DWARF', 'ELF', 'GNOME', 'HALFLING', 'HUMAN', 'TIEFLING']:
            count = len(re.findall(rf'\b{race}\b', race_text, re.IGNORECASE))
            race_mentions[race] = count

        print(f"\nRace mentions found: {race_mentions}")
else:
    print("❌ No race_text variable found. Please run the PDF extraction first.")

🔍 Debugging race parsing with your PDF text...
🔍 Debug: Race Parsing Analysis
📖 First 5 race mentions in text:

DRAGONBORN:
  Match 1: ... character, races are the lrue exolics: a hulking dragonborn here, your age could explain a particularly lowStrength or pushing his way lhrough lhe crowd, and a...
  Match 2: ...sary Born ofdragons, as their name proclaims, the dragonborn walk proudly through aworld that greets them with fearful incomprehension. Shaped bydraconic gods O...
  Match 3: ...haped bydraconic gods OI' the dragons themselves, dragonborn originally hatched from dragon eggs as aunique race, combining the best attributes ofdragons and hu...

DWARF:
  Match 1: ...ighl, isalone characlers, bul considering whyyour dwarf ischaolic. drow-a fugilive from lhe sublerranean expanse of forexample, in defiance oflawful dwarf ...
  Match 2: ...anean expanse of forexample, in defiance oflawful dwarf society can help the Underdark, trying to make his way inaworld you better define your char

In [12]:
# Add the successfully processed race chunks to ChromaDB

if 'chunks' in locals() and chunks and len(chunks) > 0:
    print("🚀 Adding race chunks to ChromaDB...")
    print("=" * 50)

    # Initialize ChromaDB manager for races
    race_chroma_manager = RaceChromaDBManager(db_path=DB_PATH, model_name=EMBEDDING_MODEL)

    # Add chunks to ChromaDB
    race_chroma_manager.add_race_chunks(chunks)

    # Show collection statistics
    print("\n📊 ChromaDB Collection Statistics:")
    stats = race_chroma_manager.get_collection_stats()
    for key, value in stats.items():
        print(f"   {key}: {value}")

    print("\n✅ Race data successfully added to ChromaDB!")
    print("   Ready for RAG queries!")

else:
    print("❌ No chunks variable found or chunks is empty")
    print("   Please run the race processing pipeline first")

🚀 Adding race chunks to ChromaDB...
🤖 Loading embedding model: all-MiniLM-L6-v2
✅ Embedding model loaded!
📚 ChromaDB initialized at: ./chromadb
   Collection: dnd_races
   Current documents: 0
📦 Adding 88 chunks to ChromaDB...
   📤 Adding batch 1/2
   📤 Adding batch 2/2
✅ Successfully added 88 race chunks to ChromaDB!
   Total documents in collection: 88

📊 ChromaDB Collection Statistics:
   total_documents: 88
   chunk_types: {'description': 23, 'traits': 9, 'names': 7, 'quick_reference': 7, 'subrace': 42}
   races: ['Gnome', 'Human', 'Duergar', 'Drow', 'Rock Gnome', 'Elf', 'Mountain Dwarf', 'Half-Elf', 'Halfling', 'Deep Gnome', 'Stout', 'High Elf', 'Forest Gnome', 'Dragonborn', 'Tiefling', 'Half-Orc', 'Dwarf', 'Wood Elf', 'Hill Dwarf']
   subraces: ['Duergar', 'Stout', 'Drow', 'High Elf', 'Rock Gnome', 'Forest Gnome', 'Mountain Dwarf', 'Wood Elf', 'Hill Dwarf', 'Deep Gnome']
   abilities_increased: {'Charisma': 5, 'Oexterity': 13}

✅ Race data successfully added to ChromaDB!
   Ready

In [14]:
# Reinitialize the Race ChromaDB Manager
# Run this if you get "race_chroma_manager not found" error

print("🔄 Reinitializing Race ChromaDB Manager...")

# Configuration (make sure these match your setup)
DB_PATH = "./chromadb"  # or "./chromadb" - match what you used before
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Create the manager (it will connect to existing collection)
race_chroma_manager = RaceChromaDBManager(db_path=DB_PATH, model_name=EMBEDDING_MODEL)

print("✅ Race ChromaDB Manager ready!")
print(f"   Collection documents: {race_chroma_manager.collection.count()}")

# Quick test to verify it's working
if race_chroma_manager.collection.count() > 0:
    print("🧪 Quick test search...")
    test_results = race_chroma_manager.search_races("elf", n_results=1)
    if test_results['documents'][0]:
        print("✅ Manager is working correctly!")
    else:
        print("⚠️ Manager connected but search returned no results")
else:
    print("⚠️ No documents found in collection - you may need to re-run the processing")

🔄 Reinitializing Race ChromaDB Manager...
🤖 Loading embedding model: all-MiniLM-L6-v2
✅ Embedding model loaded!
📚 ChromaDB initialized at: ./chromadb
   Collection: dnd_races
   Current documents: 88
✅ Race ChromaDB Manager ready!
   Collection documents: 88
🧪 Quick test search...
🔍 Searching races for: 'elf'
   📋 Found 1 results
✅ Manager is working correctly!


In [17]:
# Test the Race RAG System
# Run various searches to verify the system works

def test_race_rag_system():
    """Comprehensive test of the race RAG system"""

    try:
        # Test if manager exists and is accessible
        test_count = race_chroma_manager.collection.count()
        print(f"📊 Found {test_count} documents in race collection")
    except NameError:
        print("❌ Race ChromaDB manager not found. Please run the reinit cell first.")
        return
    except Exception as e:
        print(f"❌ Error accessing race manager: {e}")
        return

    print("🧪 Testing Race RAG System")
    print("=" * 60)

    # Test 1: Basic race search
    print("\n1️⃣ **Basic Race Search**: 'What are elves like?'")
    elf_results = race_chroma_manager.search_races("elf characteristics traits abilities", n_results=3)
    if elf_results['documents'][0]:
        for i, (doc, meta) in enumerate(zip(elf_results['documents'][0][:2], elf_results['metadatas'][0][:2])):
            print(f"\n   📄 Result {i+1}: {meta.get('race_name')} - {meta.get('chunk_type')}")
            print(f"   {doc[:200]}...")

    # Test 2: Character creation help
    print("\n\n2️⃣ **Character Creation**: 'Best race for a wizard?'")
    wizard_results = race_chroma_manager.search_races("intelligence bonus wizard spellcaster magic", n_results=3)
    if wizard_results['documents'][0]:
        for i, (doc, meta) in enumerate(zip(wizard_results['documents'][0][:2], wizard_results['metadatas'][0][:2])):
            print(f"\n   📄 Result {i+1}: {meta.get('race_name')} - {meta.get('chunk_type')}")
            bonuses = [f"{k.replace('_increase', '')}+{v}" for k, v in meta.items()
                      if k.endswith('_increase') and v > 0]
            print(f"   Bonuses: {', '.join(bonuses) if bonuses else 'None found'}")
            print(f"   {doc[:150]}...")

    # Test 3: Specific trait search
    print("\n\n3️⃣ **Trait Search**: 'Which races have darkvision?'")
    dark_results = race_chroma_manager.search_races("darkvision dark sight", n_results=5, has_darkvision=True)
    if dark_results['documents'][0]:
        darkvision_races = set()
        for meta in dark_results['metadatas'][0]:
            if meta.get('has_darkvision'):
                race_name = meta.get('race_name', 'Unknown')
                darkvision_range = meta.get('darkvision_range', 0)
                darkvision_races.add(f"{race_name} ({darkvision_range}ft)")

        print(f"   🌙 Races with darkvision: {', '.join(sorted(darkvision_races))}")

    # Test 4: Subrace information
    print("\n\n4️⃣ **Subrace Search**: 'Tell me about dwarf subraces'")
    dwarf_results = race_chroma_manager.get_race_by_name("Dwarf")
    if dwarf_results['documents'][0]:
        subraces = set()
        for meta in dwarf_results['metadatas'][0]:
            if meta.get('chunk_type') == 'subrace':
                subraces.add(meta.get('race_name', 'Unknown'))

        print(f"   ⛏️  Dwarf subraces found: {', '.join(sorted(subraces))}")

        # Show one subrace example
        for doc, meta in zip(dwarf_results['documents'][0], dwarf_results['metadatas'][0]):
            if meta.get('chunk_type') == 'subrace':
                print(f"\n   📄 Example - {meta.get('race_name')}:")
                print(f"   {doc[:200]}...")
                break

    # Test 5: Cultural/lore search
    print("\n\n5️⃣ **Lore Search**: 'Forest dwelling races'")
    forest_results = race_chroma_manager.search_races("forest woodland nature tree", n_results=3)
    if forest_results['documents'][0]:
        for i, (doc, meta) in enumerate(zip(forest_results['documents'][0][:2], forest_results['metadatas'][0][:2])):
            print(f"\n   📄 Result {i+1}: {meta.get('race_name')} - {meta.get('chunk_type')}")
            print(f"   {doc[:150]}...")

# Run the comprehensive test
try:
    test_race_rag_system()
except NameError:
    print("❌ Please run the 'Reinitialize Race ChromaDB Manager' cell first!")
except Exception as e:
    print(f"❌ Error running tests: {e}")

print("\n" + "="*60)
print("🎯 **Race RAG System Ready!**")
print("Available functions:")
print("- race_chroma_manager.search_races(query, n_results)")
print("- race_chroma_manager.get_race_by_name(race_name)")
print("- character_creation_helper(class_name)")
print("- dm_encounter_helper(environment, party_level)")

📊 Found 88 documents in race collection
🧪 Testing Race RAG System

1️⃣ **Basic Race Search**: 'What are elves like?'
🔍 Searching races for: 'elf characteristics traits abilities'
   📋 Found 3 results

   📄 Result 1: Elf - quick_reference
   **Elf - Quick Reference:**

**Key Features:**
- Darkvision (60 ft)
- Languages: Common
- Base Speed: Your base walkiog speed is30feel
- Size Category: Elves raoge from under 5toover 6feet tall aod hav...

   📄 Result 2: Drow - subrace
   **Drow** (Subrace of Elf)

Drow advenlurers are rare, and lhe race does nol exisl elves (also called wild elves. green elves, or foresl elves) inali worlds. Check wilh your Dungeon Masler losee are re...


2️⃣ **Character Creation**: 'Best race for a wizard?'
🔍 Searching races for: 'intelligence bonus wizard spellcaster magic'
   📋 Found 3 results

   📄 Result 1: Drow - subrace
   Bonuses: None found
   **Drow** (Subrace of Elf)

drow characler. Wood elves' skin lends lobe copperish inhue, Ability Score Increase. Yo